# Question 1 : Multi-Task CNN on Fashion-MNIST

### Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from IPython.display import display, Markdown
import os
# os.environ['WANDB_CONSOLE'] = 'off'
# os.environ['WANDB_DISABLE_CODE'] = 'true'
# os.environ['WANDB_SILENT'] = 'true'
import wandb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, f1_score
from typing import Tuple, Dict, List
from datetime import datetime
import pandas as pd
from tqdm import tqdm

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Some preliminary code
def add_username(username) -> None:
    plt.text(0.95, 0.95, username, ha="right", va="top", transform=plt.gca().transAxes, fontsize=10, color="gray", alpha=0.7)

def print_separator() -> None:
    print("="*50)

In [ ]:
# Print the cuda device name
if torch.cuda.is_available():
    print(f"CUDA Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Available GPUs: {torch.cuda.device_count()}")

### Dataset Class

In [ ]:
class FashionMNISTDataset(Dataset):
    """
    Custom Dataset class for Fashion-MNIST with classification and regression targets.
    
    Returns:
        image: normalized tensor [1, 28, 28]
        class_label: integer label in {0, ..., 9}
        ink_target: normalized pixel intensity (average pixel value)
    """

    def __init__(self, data, targets, transform=None, mean=None, std=None):
        """
        Initialize the dataset.
        
        Args:
            data: Image data
            targets: Classification labels
            transform: Data augmentation transforms
        """
        self.data = data
        self.targets = targets
        self.transform = transform
        
        if mean is None or std is None:
            raise ValueError("mean and std must be provided for normalization")

        self.mean = mean
        self.std = std
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Get image and label
        image = self.data[idx]
        class_label = self.targets[idx]
        
        # Convert to PIL Image for transforms
        image_pil = transforms.ToPILImage()(image)
        
        # Calculate ink target
        image_np = np.array(image_pil).astype(np.float32) / 255.0
        ink_target = float(np.mean(image_np))
        
        # Apply transforms if any
        if self.transform:
            image_tensor = self.transform(image_pil)
        else:
            # Default: just convert to tensor and normalize
            image_tensor = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[self.mean], std=[self.std])
            ])(image_pil)
        
        class_label = int(class_label)
        ink_target = np.float32(ink_target)
        
        return image_tensor, class_label, ink_target

def calculate_mean_std(data):
    """
    Calculate mean and std of the dataset.
    
    Args:
        data: Tensor of images [N, H, W]
    
    Returns:
        mean, std
    """
    # Convert to float and normalize to [0, 1]
    data_float = data.float() / 255.0
    
    # Calculate mean and std across all pixels
    mean = data_float.mean().item()
    std = data_float.std().item()
    
    return mean, std


def load_fashion_data(batch_size=64, train_val_split=0.9):
    """
    Load Fashion-MNIST dataset and create train/val/test splits.
    
    Args:
        batch_size: Batch size for dataloaders
        train_val_split: Ratio for train/val split (default 90/10)
    
    Returns:
        train_loader, val_loader, test_loader
    """
    # Download raw Fashion-MNIST data
    train_data_raw = FashionMNIST(root='./data', train=True, download=True)
    test_data_raw = FashionMNIST(root='./data', train=False, download=True)
    
    # Extract data and targets
    train_images = train_data_raw.data
    train_labels = train_data_raw.targets
    test_images = test_data_raw.data
    test_labels = test_data_raw.targets

    print_separator()
    print("Calculating dataset statistics")
    mean, std = calculate_mean_std(train_images)
    print(f"Dataset mean: {mean:.4f}, std: {std:.4f}")
    print_separator()
    
    # Split train into train and validation
    total_train = len(train_images)
    train_size = int(train_val_split * total_train)
    val_size = total_train - train_size
    
    indices = torch.randperm(total_train).tolist()
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]
    
    train_images_split = train_images[train_indices]
    train_labels_split = train_labels[train_indices]
    val_images_split = train_images[val_indices]
    val_labels_split = train_labels[val_indices]
    
    # Define augmentations for training
    train_transform = transforms.Compose([
        transforms.RandomCrop(28, padding=2),  # Random crop with padding
        transforms.RandomRotation(10),  # Small rotation
        transforms.ToTensor(),
        transforms.Normalize(mean=[mean], std=[std])
    ])
    
    # No augmentation for validation and test
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[mean], std=[std])
    ])
    
    # Create datasets
    train_dataset = FashionMNISTDataset(train_images_split, train_labels_split, transform=train_transform, mean=mean, std=std)
    val_dataset = FashionMNISTDataset(val_images_split, val_labels_split, transform=eval_transform, mean=mean, std=std)
    test_dataset = FashionMNISTDataset(test_images, test_labels, transform=eval_transform, mean=mean, std=std)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    print(f"Train size: {len(train_dataset)}")
    print(f"Validation size: {len(val_dataset)}")
    print(f"Test size: {len(test_dataset)}")
    
    return train_loader, val_loader, test_loader

### Model Architecture

In [ ]:
class MultiTaskCNN(nn.Module):
    """
    Multi-task CNN for Fashion-MNIST classification and ink regression.
    
    Architecture:
    - Shared convolutional backbone (2-3 Conv-BatchNorm-ReLU-Pool blocks with dropout)
    - Classification head: outputs logits [B, 10]
    - Regression head: outputs scalar values [B, 1]
    """
    
    def __init__(self, num_classes=10, dropout_rate=0.3, num_filters=32):
        """
        Initialize the multi-task CNN.
        
        Args:
            num_classes: Number of output classes (10 for Fashion-MNIST)
            dropout_rate: Dropout probability
            num_filters: Base number of filters in first conv layer
        """
        super().__init__()
        
        # Shared convolutional backbone
        # Block 1: [B, 1, 28, 28] -> [B, num_filters, 14, 14]
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, num_filters, kernel_size=3, padding=1), # [IC, OC, ...],
            nn.BatchNorm2d(num_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate)
        )
        
        # Block 2: [B, num_filters, 14, 14] -> [B, num_filters*2, 7, 7]
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(num_filters, num_filters*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_filters*2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate)
        )
        
        # Block 3: [B, num_filters*2, 7, 7] -> [B, num_filters*4, 3, 3]
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(num_filters*2, num_filters*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_filters*4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate)
        )
        
        # Calculate flattened size: num_filters*4 * 3 * 3
        flattened_size = num_filters * 4 * 3 * 3
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )
        
        # Regression head
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Store intermediate outputs for visualization
        self.intermediate_outputs = {}
    
    def forward(self, x, return_features=False):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor [B, 1, 28, 28]
            return_features: If True, return intermediate feature maps
        
        Returns:
            classification_logits: [B, 10]
            regression_output: [B, 1]
            features (optional): dict of intermediate feature maps
        """
        # Shared backbone
        x1 = self.conv_block1(x)
        x2 = self.conv_block2(x1)
        x3 = self.conv_block3(x2)
        
        # Store intermediate outputs
        if return_features:
            self.intermediate_outputs = {
                'conv_block1': x1.detach().cpu(),
                'conv_block2': x2.detach().cpu(),
                'conv_block3': x3.detach().cpu()
            }
        
        # Task-specific heads
        classification_logits = self.classifier(x3)
        regression_output = self.regressor(x3)
        
        if return_features:
            return classification_logits, regression_output, self.intermediate_outputs
        
        return classification_logits, regression_output


def joint_loss(class_logits, class_targets, reg_outputs, reg_targets, 
               lambda1=1.0, lambda2=1.0):
    """
    Compute joint loss for multi-task learning.
    
    L_total = λ1 * L_CE + λ2 * L_MSE
    
    Args:
        class_logits: Classification logits [B, 10]
        class_targets: True class labels [B]
        reg_outputs: Regression predictions [B, 1]
        reg_targets: True ink values [B]
        lambda1: Weight for classification loss
        lambda2: Weight for regression loss
    
    Returns:
        total_loss, ce_loss, mse_loss
    """
    ce_loss = nn.CrossEntropyLoss()(class_logits, class_targets)
    mse_loss = nn.MSELoss()(reg_outputs.squeeze(), reg_targets)
    
    total_loss = lambda1 * ce_loss + lambda2 * mse_loss
    
    return total_loss, ce_loss, mse_loss

### Train and Evaluate

In [ ]:
def train_epoch(model, train_loader, optimizer, lambda1, lambda2, device):
    """
    Train for one epoch.
    
    Returns:
        Dictionary with training metrics
    """
    model.train()
    
    total_loss = 0.0
    total_ce_loss = 0.0
    total_mse_loss = 0.0
    all_preds = []
    all_targets = []
    all_reg_preds = []
    all_reg_targets = []

    pbar = tqdm(train_loader, desc='Training', leave=False)
    
    for images, class_labels, ink_targets in pbar:
        images = images.to(device)
        class_labels = class_labels.to(device)
        ink_targets = ink_targets.to(device)
        
        # Forward pass
        class_logits, reg_outputs = model(images)
        
        # Compute loss
        loss, ce_loss, mse_loss = joint_loss(
            class_logits, class_labels, reg_outputs, ink_targets,
            lambda1, lambda2
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate metrics
        total_loss += loss.item() * images.size(0)
        total_ce_loss += ce_loss.item() * images.size(0)
        total_mse_loss += mse_loss.item() * images.size(0)
        
        # Store predictions
        preds = torch.argmax(class_logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(class_labels.cpu().numpy())
        all_reg_preds.extend(reg_outputs.squeeze().detach().cpu().numpy())
        all_reg_targets.extend(ink_targets.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    # Calculate metrics
    n_samples = len(train_loader.dataset)
    metrics = {
        'loss': total_loss / n_samples,
        'ce_loss': total_ce_loss / n_samples,
        'mse_loss': total_mse_loss / n_samples,
        'accuracy': accuracy_score(all_targets, all_preds),
        'f1_score': f1_score(all_targets, all_preds, average='weighted'),
        'mae': mean_absolute_error(all_reg_targets, all_reg_preds),
        'rmse': np.sqrt(mean_squared_error(all_reg_targets, all_reg_preds))
    }
    
    return metrics


def evaluate(model, data_loader, lambda1, lambda2, device):
    """
    Evaluate model on validation or test set.
    
    Returns:
        Dictionary with evaluation metrics
    """
    model.eval()
    
    total_loss = 0.0
    total_ce_loss = 0.0
    total_mse_loss = 0.0
    all_preds = []
    all_targets = []
    all_reg_preds = []
    all_reg_targets = []

    pbar = tqdm(data_loader, desc='Validation', leave=False)
    
    with torch.no_grad():
        for images, class_labels, ink_targets in pbar:
            images = images.to(device)
            class_labels = class_labels.to(device)
            ink_targets = torch.tensor(ink_targets, dtype=torch.float32).to(device)
            
            # Forward pass
            class_logits, reg_outputs = model(images)
            
            # Compute loss
            loss, ce_loss, mse_loss = joint_loss(
                class_logits, class_labels, reg_outputs, ink_targets,
                lambda1, lambda2
            )
            
            # Accumulate metrics
            total_loss += loss.item() * images.size(0)
            total_ce_loss += ce_loss.item() * images.size(0)
            total_mse_loss += mse_loss.item() * images.size(0)
            
            # Store predictions
            preds = torch.argmax(class_logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(class_labels.cpu().numpy())
            all_reg_preds.extend(reg_outputs.squeeze().cpu().numpy())
            all_reg_targets.extend(ink_targets.cpu().numpy())
    
    # Calculate metrics
    n_samples = len(data_loader.dataset)
    metrics = {
        'loss': total_loss / n_samples,
        'ce_loss': total_ce_loss / n_samples,
        'mse_loss': total_mse_loss / n_samples,
        'accuracy': accuracy_score(all_targets, all_preds),
        'f1_score': f1_score(all_targets, all_preds, average='weighted'),
        'mae': mean_absolute_error(all_reg_targets, all_reg_preds),
        'rmse': np.sqrt(mean_squared_error(all_reg_targets, all_reg_preds))
    }
    
    return metrics


def train_model(model, train_loader, val_loader, config, device):
    """
    Complete training loop with wandb logging.
    
    Args:
        model: MultiTaskCNN model
        train_loader: Training data loader
        val_loader: Validation data loader
        config: Dictionary with hyperparameters
        device: Device to train on
    
    Returns:
        Trained model and training history
    """
    # Initialize optimizer
    # weight_decay is the L2 regularization strength
    if config['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=0.9, weight_decay=1e-4)
    elif config['optimizer'] == 'Adam': #(Adaptive Moment Estimation) - Uses adaptive LR
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=1e-4)
    elif config['optimizer'] == 'AdamW': #(Adam with a more effective Weight Decay)
        optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-4)
    
    # Learning rate scheduler - acts at a global scale unlike Adam which acts at a per-parameter level
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    best_val_acc = 0.0
    best_val_rmse = float('inf')
    history = {
        'train_loss': [], 'train_ce': [], 'train_mse': [], 'train_acc': [], 
        'train_mae': [], 'train_rmse': [], 'train_f1': [],
        'val_loss': [], 'val_ce': [], 'val_mse': [], 'val_acc': [], 
        'val_mae': [], 'val_rmse': [], 'val_f1': []
    }
    
    # Progress bar for epochs
    epoch_pbar = tqdm(range(config['epochs']), desc=f"Training {config.get('name', 'model')}")
    
    for epoch in epoch_pbar:
        # Train
        train_metrics = train_epoch(model, train_loader, optimizer, config['lambda1'], config['lambda2'], device)
        
        # Validate
        val_metrics = evaluate(model, val_loader, config['lambda1'], config['lambda2'], device)

        # Update scheduler
        scheduler.step(val_metrics['loss'])
        
        # Store history
        history['train_loss'].append(train_metrics['loss'])
        history['train_ce'].append(train_metrics['ce_loss'])
        history['train_mse'].append(train_metrics['mse_loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['train_mae'].append(train_metrics['mae'])
        history['train_rmse'].append(train_metrics['rmse'])
        history['train_f1'].append(train_metrics['f1_score'])

        history['val_loss'].append(val_metrics['loss'])
        history['val_ce'].append(val_metrics['ce_loss'])
        history['val_mse'].append(val_metrics['mse_loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_mae'].append(val_metrics['mae'])
        history['val_rmse'].append(val_metrics['rmse'])
        history['val_f1'].append(val_metrics['f1_score'])
        
        # Log to wandb
        wandb.log({
            'epoch': epoch + 1,
            'train/loss': train_metrics['loss'],
            'train/ce_loss': train_metrics['ce_loss'],
            'train/mse_loss': train_metrics['mse_loss'],
            'train/accuracy': train_metrics['accuracy'],
            'train/mae': train_metrics['mae'],
            'train/rmse': train_metrics['rmse'],
            'train/f1_score': train_metrics['f1_score'],
            'val/loss': val_metrics['loss'],
            'val/ce_loss': val_metrics['ce_loss'],
            'val/mse_loss': val_metrics['mse_loss'],
            'val/accuracy': val_metrics['accuracy'],
            'val/mae': val_metrics['mae'],
            'val/rmse': val_metrics['rmse'],
            'val/f1_score': val_metrics['f1_score'],
            'learning_rate': optimizer.param_groups[0]['lr']
        })
        
        # Track best models
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
        if val_metrics['rmse'] < best_val_rmse:
            best_val_rmse = val_metrics['rmse']
        
        # Update epoch progress bar with metrics
        epoch_pbar.set_postfix({
            'train_acc': f"{train_metrics['accuracy']:.3f}",
            'val_acc': f"{val_metrics['accuracy']:.3f}",
            'train_rmse': f"{train_metrics['rmse']:.3f}",
            'val_rmse': f"{val_metrics['rmse']:.3f}",
            'lr': f"{optimizer.param_groups[0]['lr']:.6f}"
        })
    
    # Print final summary
    print(f"\nTraining complete!")
    print(f"  Best Val Accuracy: {best_val_acc:.4f}")
    print(f"  Best Val RMSE: {best_val_rmse:.4f}")

    return model, history

### Visualizations

In [ ]:
def visualize_feature_maps(model, test_loader, device, num_images=3):
    """
    Visualize intermediate feature maps for test images.
    
    Args:
        model: Trained model
        test_loader: Test data loader
        device: Device
        num_images: Number of test images to visualize
    """
    model.eval()
    
    # Get a batch of test images
    images, labels, _ = next(iter(test_loader))
    images = images[:num_images].to(device)
    
    # Get feature maps
    with torch.no_grad():
        _, _, features = model(images, return_features=True)
    
    # Create figure
    fig, axes = plt.subplots(num_images, 4, figsize=(16, 4*num_images))
    
    class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
                   'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
    
    for img_idx in range(num_images):
        # Original image
        img = images[img_idx].cpu().squeeze()
        img = (img * 0.3530) + 0.2860  # Denormalize
        axes[img_idx, 0].imshow(img, cmap='gray')
        axes[img_idx, 0].set_title(f'Original: {class_names[labels[img_idx]]}')
        axes[img_idx, 0].axis('off')
        
        # Feature maps from each conv block
        for block_idx, block_name in enumerate(['conv_block1', 'conv_block2', 'conv_block3']):
            feature_map = features[block_name][img_idx]
            
            # Average across all channels
            avg_feature = torch.mean(feature_map, dim=0).numpy()
            
            axes[img_idx, block_idx+1].imshow(avg_feature, cmap='viridis')
            axes[img_idx, block_idx+1].set_title(f'{block_name}')
            axes[img_idx, block_idx+1].axis('off')
    
    plt.tight_layout()
    add_username("siddarth.g")

    run = wandb.init(
        project="q1-fashion-mnist-multitask",
        name=f"siddarth.g_{exp_config['name']}",
        config=exp_config,
        reinit=True
    )
    # Save figure
    wandb.log({"feature_maps": wandb.Image(fig)})
    wandb.finish()
    
    plt.show()
    plt.close()
    
    print("\n=== Feature Map Interpretation ===")
    print("Conv Block 1 (Early layers):")
    print("  - Captures low-level features: edges, corners, simple textures")
    print("  - Detects basic shapes and contours of clothing items")
    print("  - Responds to intensity gradients important for ink regression")
    
    print("\nConv Block 2 (Middle layers):")
    print("  - Combines edges into more complex patterns")
    print("  - Detects clothing-specific textures (fabric patterns, stitching)")
    print("  - Begins to recognize parts of garments (collars, sleeves, straps)")
    
    print("\nConv Block 3 (Deep layers):")
    print("  - Captures high-level semantic features")
    print("  - Recognizes specific clothing regions and overall garment structure")
    print("  - Provides abstract representations useful for both classification and ink prediction")
    print("  - Helps distinguish between similar categories (shirts vs. t-shirts)")


def plot_training_curves(history, title_suffix=""):
    """
    Plot training and validation curves.
    
    Args:
        history: Dictionary with training history
        title_suffix: Additional text for title
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Total loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Total Loss')
    axes[0, 0].set_title(f'Total Loss {title_suffix}')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Classification loss
    axes[0, 1].plot(epochs, history['train_ce'], 'b-', label='Train', linewidth=2)
    axes[0, 1].plot(epochs, history['val_ce'], 'r-', label='Validation', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('CE Loss')
    axes[0, 1].set_title(f'Classification Loss (CE) {title_suffix}')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Regression loss
    axes[0, 2].plot(epochs, history['train_mse'], 'b-', label='Train', linewidth=2)
    axes[0, 2].plot(epochs, history['val_mse'], 'r-', label='Validation', linewidth=2)
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('MSE Loss')
    axes[0, 2].set_title(f'Regression Loss (MSE) {title_suffix}')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, 0].plot(epochs, history['train_acc'], 'b-', label='Train', linewidth=2)
    axes[1, 0].plot(epochs, history['val_acc'], 'r-', label='Validation', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title(f'Classification Accuracy {title_suffix}')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # MAE
    axes[1, 1].plot(epochs, history['train_mae'], 'b-', label='Train', linewidth=2)
    axes[1, 1].plot(epochs, history['val_mae'], 'r-', label='Validation', linewidth=2)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('MAE')
    axes[1, 1].set_title(f'Regression MAE {title_suffix}')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # RMSE
    axes[1, 2].plot(epochs, history['train_rmse'], 'b-', label='Train', linewidth=2)
    axes[1, 2].plot(epochs, history['val_rmse'], 'r-', label='Validation', linewidth=2)
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('RMSE')
    axes[1, 2].set_title(f'Regression RMSE {title_suffix}')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    add_username("siddarth.g")
    
    wandb.log({f"training_curves_{title_suffix}": wandb.Image(fig)})
    plt.show()
    plt.close()

### Function Calls

In [ ]:
# Load Fashion-MNIST data
print("Loading Fashion-MNIST dataset...")
train_loader, val_loader, test_loader = load_fashion_data(batch_size=64, train_val_split=0.9)
print("Data loaded successfully!\n")

In [ ]:
# Define sweep configuration
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'val/accuracy',
        'goal': 'maximize'
    },
    'parameters': {
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-4,
            'max': 1e-2
        },
        'optimizer': {
            'values': ['SGD', 'Adam', 'AdamW']
        },
        'dropout': {
            'values': [0.1, 0.2, 0.3, 0.4]
        },
        'num_filters': {
            'values': [16, 32, 64]
        },
        'batch_size': {
            'values': [32, 64, 128, 256]
        },
        'lambda1': {
            'values': [0.5, 1.0, 2.0]
        },
        'lambda2': {
            'values': [0.5, 1.0, 2.0]
        },
        'epochs': {
            'value': 30
        }
    }
}

# Login and create sweep
wandb.login(key='f5bbdbea853efc716c2b2318ba8efaf2f4a6909e')
SWEEP_ID = wandb.sweep(sweep=sweep_config, project="q1-fashion-mnist-multitask")
sweep_id = SWEEP_ID
print(f"Sweep created with ID: {sweep_id}")

In [ ]:
def train_with_sweep():
    """Training function for sweep - with Jupyter compatibility fixes."""
    
    # Initialize with specific settings for Jupyter
    run = wandb.init()
    
    config = wandb.config
    
    print_separator()
    print(f"Starting run: {run.name}")
    print(f"  lr={config.lr:.6f}, opt={config.optimizer}, dropout={config.dropout}")
    print(f"  filters={config.num_filters}, batch={config.batch_size}")
    print(f"  lambda1={config.lambda1}, lambda2={config.lambda2}")
    print_separator()
    
    # Load data
    train_loader, val_loader, test_loader = load_fashion_data(
        batch_size=config.batch_size,
        train_val_split=0.9
    )
    
    # Create model
    model = MultiTaskCNN(
        num_classes=10,
        dropout_rate=config.dropout,
        num_filters=config.num_filters
    )

    # if torch.cuda.device_count() > 1:
    #     print(f"Using {torch.cuda.device_count()} GPUs!")
    #     model = torch.nn.DataParallel(model)

    model = model.to(DEVICE)
    
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model parameters: {num_params:,}")
    
    # Training config
    training_config = {
        'lr': config.lr,
        'optimizer': config.optimizer,
        'dropout': config.dropout,
        'num_filters': config.num_filters,
        'lambda1': config.lambda1,
        'lambda2': config.lambda2,
        'batch_size': config.batch_size,
        'epochs': config.epochs,
        'name': run.name
    }
    
    # Train
    model, history = train_model(model, train_loader, val_loader, training_config, DEVICE)
    
    # Test
    test_metrics = evaluate(model, test_loader, config.lambda1, config.lambda2, DEVICE)
    
    print(f"\nTest: Acc={test_metrics['accuracy']:.4f}, F1={test_metrics['f1_score']:.4f}, RMSE={test_metrics['rmse']:.4f}")
    
    # Log test metrics
    wandb.log({
        'test/accuracy': test_metrics['accuracy'],
        'test/f1_score': test_metrics['f1_score'],
        'test/mae': test_metrics['mae'],
        'test/rmse': test_metrics['rmse']
    })
    
    # Summary
    wandb.run.summary['best_val_accuracy'] = max(history['val_acc'])
    wandb.run.summary['best_val_f1'] = max(history['val_f1'])
    wandb.run.summary['best_val_rmse'] = min(history['val_rmse'])
    wandb.run.summary['final_test_accuracy'] = test_metrics['accuracy']
    wandb.run.summary['final_test_f1'] = test_metrics['f1_score']
    wandb.run.summary['final_test_rmse'] = test_metrics['rmse']
        
    wandb.finish()

In [ ]:
print(f"Starting sweep agent for sweep: {SWEEP_ID}")

wandb.agent(SWEEP_ID, function=train_with_sweep, count=10)

print("\nSweep completed successfully!")

In [ ]:
# Get sweep results from wandb API
api = wandb.Api()
sweep = api.sweep(f"{wandb.api.default_entity}/q1-fashion-mnist-multitask/{sweep_id}")

# Get all runs from this sweep
runs = sweep.runs

# Collect results
results = []
for run in runs:
    if run.state == "finished":
        results.append({
            'name': run.name,
            'id': run.id,
            'lr': run.config.get('lr'),
            'optimizer': run.config.get('optimizer'),
            'dropout': run.config.get('dropout'),
            'num_filters': run.config.get('num_filters'),
            'batch_size': run.config.get('batch_size'),
            'lambda1': run.config.get('lambda1'),
            'lambda2': run.config.get('lambda2'),
            'val_accuracy': run.summary.get('best_val_accuracy'),
            'val_rmse': run.summary.get('best_val_rmse'),
            'test_accuracy': run.summary.get('final_test_accuracy'),
            'test_rmse': run.summary.get('final_test_rmse'),
            'val_f1': run.summary.get('best_val_f1'),
            'test_f1': run.summary.get('final_test_f1'),
            'url': run.url
        })

# Sort by validation accuracy
results_sorted = sorted(results, key=lambda x: x['val_accuracy'], reverse=True)

print("\n=== SWEEP RESULTS ===\n")
print("Top 5 Models by Validation Accuracy:")
print_separator()
for i, result in enumerate(results_sorted[:5], 1):
    print(f"{i}. {result['name']}")
    print(f"   Val Acc: {result['val_accuracy']:.4f} | Test Acc: {result['test_accuracy']:.4f}")
    print(f"   Val RMSE: {result['val_rmse']:.4f} | Test RMSE: {result['test_rmse']:.4f}")
    print(f"   Config: lr={result['lr']:.6f}, opt={result['optimizer']}, lambda1={result['lambda1']}, lambda2={result['lambda2']}")
    print(f"   Val Acc: {result['val_accuracy']:.4f} | Val F1: {result['val_f1']:.4f}")
    print(f"   Test Acc: {result['test_accuracy']:.4f} | Test F1: {result['test_f1']:.4f}")
    print(f"   URL: {result['url']}")
    print()

# Sort by validation RMSE
results_sorted_rmse = sorted(results, key=lambda x: x['val_rmse'])

print("\nTop 5 Models by Validation RMSE:")
print_separator()
for i, result in enumerate(results_sorted_rmse[:5], 1):
    print(f"{i}. {result['name']}")
    print(f"   Val RMSE: {result['val_rmse']:.4f} | Test RMSE: {result['test_rmse']:.4f}")
    print(f"   Val Acc: {result['val_accuracy']:.4f} | Test Acc: {result['test_accuracy']:.4f}")
    print(f"   Config: lr={result['lr']:.6f}, opt={result['optimizer']}, lambda1={result['lambda1']}, lambda2={result['lambda2']}")
    print(f"   URL: {result['url']}")
    print()

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('val_accuracy', ascending=False)

print("\n=== FULL RESULTS TABLE ===")
print(df_results[['name', 'lr', 'optimizer', 'lambda1', 'lambda2', 
                  'val_accuracy', 'test_accuracy', 'val_rmse', 'test_rmse']].to_string(index=False))

# Best models
best_acc_model = results_sorted[0]
best_rmse_model = results_sorted_rmse[0]

print_separator()
print("=== BEST MODELS ===")
print_separator()
print(f"\n Best Classification Model: {best_acc_model['name']}")
print(f"   Val Accuracy: {best_acc_model['val_accuracy']:.4f}")
print(f"   Test Accuracy: {best_acc_model['test_accuracy']:.4f}")
print(f"   lambda1={best_acc_model['lambda1']}, lambda2={best_acc_model['lambda2']}")

print(f"\n Best Regression Model: {best_rmse_model['name']}")
print(f"   Val RMSE: {best_rmse_model['val_rmse']:.4f}")
print(f"   Test RMSE: {best_rmse_model['test_rmse']:.4f}")
print(f"   lambda1={best_rmse_model['lambda1']}, lambda2={best_rmse_model['lambda2']}")

print_separator()
print(f"Full sweep dashboard: https://wandb.ai/{wandb.api.default_entity}/q1-fashion-mnist-multitask/sweeps/{sweep_id}")
print_separator()

In [ ]:
df_results = pd.DataFrame(results)
    
# Find best models
best_acc_idx = df_results['val_accuracy'].idxmax()
best_rmse_idx = df_results['val_rmse'].idxmin()

best_acc_model = df_results.loc[best_acc_idx]
best_rmse_model = df_results.loc[best_rmse_idx]

# NOTE: Feature map visualization requires the actual model object
# Since we're loading results from wandb API, the models aren't available in memory
# To visualize feature maps, you would need to:
# 1. Save model checkpoints during training with wandb.save()
# 2. Load the model from wandb artifacts here
# 3. Then call visualize_feature_maps()

# Commented out - model object not available from API results
# print("\n=== FEATURE MAP VISUALIZATION ===")
# print(f"Using model: {results[best_acc_idx]['name']}\n")
# visualize_feature_maps(
#     results[best_acc_idx]['model'], 
#     test_loader, 
#     DEVICE, 
#     num_images=3
# )

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
# 1. Lambda1 vs Accuracy
scatter1 = axes[0, 0].scatter(df_results['lambda1'], df_results['test_accuracy'], 
                                c=df_results['lambda2'], cmap='viridis', s=100, alpha=0.7)
axes[0, 0].set_xlabel('lambda1 (Classification Weight)')
axes[0, 0].set_ylabel('Test Accuracy')
axes[0, 0].set_title('Effect of lambda1 on Classification')
axes[0, 0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0, 0], label='lambda2')

# 2. Lambda2 vs RMSE
scatter2 = axes[0, 1].scatter(df_results['lambda2'], df_results['test_rmse'], 
                                c=df_results['lambda1'], cmap='viridis', s=100, alpha=0.7)
axes[0, 1].set_xlabel('lambda2 (Regression Weight)')
axes[0, 1].set_ylabel('Test RMSE')
axes[0, 1].set_title('Effect of lambda2 on Regression')
axes[0, 1].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[0, 1], label='lambda1')

# 3. Accuracy vs RMSE trade-off
axes[0, 2].scatter(df_results['test_accuracy'], df_results['test_rmse'], s=100, alpha=0.7)
axes[0, 2].scatter(best_acc_model['test_accuracy'], best_acc_model['test_rmse'], 
                    c='red', s=200, marker='*', label='Best Acc', edgecolors='black', linewidths=2)
axes[0, 2].scatter(best_rmse_model['test_accuracy'], best_rmse_model['test_rmse'], 
                    c='green', s=200, marker='*', label='Best RMSE', edgecolors='black', linewidths=2)
axes[0, 2].set_xlabel('Test Accuracy')
axes[0, 2].set_ylabel('Test RMSE')
axes[0, 2].set_title('Classification vs Regression Trade-off')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# 4. Optimizer comparison
optimizer_acc = df_results.groupby('optimizer')['test_accuracy'].mean()
axes[1, 0].bar(optimizer_acc.index, optimizer_acc.values, alpha=0.7, color=['skyblue', 'salmon', 'lightgreen'])
axes[1, 0].set_xlabel('Optimizer')
axes[1, 0].set_ylabel('Average Test Accuracy')
axes[1, 0].set_title('Optimizer Performance')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. Learning rate vs accuracy
axes[1, 1].scatter(df_results['lr'], df_results['test_accuracy'], s=100, alpha=0.7)
axes[1, 1].set_xlabel('Learning Rate')
axes[1, 1].set_ylabel('Test Accuracy')
axes[1, 1].set_title('Learning Rate Impact')
axes[1, 1].set_xscale('log')
axes[1, 1].grid(True, alpha=0.3)

# 6. Lambda ratio analysis
df_results['lambda_ratio'] = df_results['lambda1'] / df_results['lambda2']
axes[1, 2].scatter(df_results['lambda_ratio'], df_results['test_accuracy'], 
                    label='Accuracy', alpha=0.7, s=100)
ax2 = axes[1, 2].twinx()
ax2.scatter(df_results['lambda_ratio'], df_results['test_rmse'], 
            color='orange', label='RMSE', alpha=0.7, s=100)
axes[1, 2].set_xlabel('lambda1 / lambda2 Ratio')
axes[1, 2].set_ylabel('Test Accuracy', color='blue')
ax2.set_ylabel('Test RMSE', color='orange')
axes[1, 2].set_title('Lambda Balance Effect')
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
add_username("siddarth.g")
plt.show()

In [ ]:
print_separator()
print("=== ANALYSIS: EFFECT OF lambda1 AND lambda2 ON MULTI-TASK TRADE-OFF ===")
print_separator()

print("Impact of Loss Weighting on Model Performance:\n")

print("1. Balanced Weights (lambda1=1.0, lambda2=1.0):")
print("   - Provides a good balance between both tasks")
print("   - Neither task dominates the learning process")
print("   - Suitable as a baseline configuration\n")

print("2. Higher Classification Weight (lambda1=2.0, lambda2=1.0):")
print("   - Model prioritizes classification accuracy")
print("   - May achieve higher accuracy on classification task")
print("   - Regression performance might be slightly compromised")
print("   - The shared backbone learns features more suited for classification\n")

print("3. Higher Regression Weight (lambda1=1.0, lambda2=2.0):")
print("   - Model focuses more on predicting ink values accurately")
print("   - May achieve lower RMSE/MAE")
print("   - Classification accuracy might decrease slightly")
print("   - The backbone learns features that capture pixel intensity patterns\n")

print("Key Observations:")
print("- The joint loss creates a trade-off between the two objectives")
print("- Adjusting lambda1 and lambda2 controls which task receives more gradient signal")
print("- The shared backbone must learn features useful for both tasks")
print("- Extreme weighting can lead to catastrophic forgetting of one task")
print("- Optimal weights depend on the relative importance of each task\n")

print("Practical Considerations:")
print("- If classification is more critical, use higher lambda1")
print("- If ink prediction is more important, use higher lambda2")
print("- Multi-task learning can improve generalization through shared representations")
print("- The auxiliary regression task can act as a regularizer for classification\n")

print_separator()